## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### Deriving an Index Set via `union` (Diet Model Pattern)
#### AMPL Formulation

In [4]:
%%ampl_eval
reset;              # clear previous model/data

# Nutrient sets: lower-bound and upper-bound requirements
set MINREQ;                           # nutrients with a minimum daily requirement
set MAXREQ;                           # nutrients with a maximum daily requirement

set NUTR = MINREQ union MAXREQ;       # derived full nutrient set (union of both)
set FOOD;                             # food items

param cost  {FOOD}  > 0;              # cost per unit of each food
param f_min {FOOD}  >= 0, default 0;  # minimum servings of each food
param f_max {j in FOOD} >= f_min[j], default 10;  # maximum servings of each food
param n_min {MINREQ} >= 0;            # minimum nutrient amounts
param n_max {MAXREQ} >= 0;            # maximum nutrient amounts
param amt   {NUTR, FOOD} >= 0;        # nutrient amount per unit of food

var Buy {j in FOOD} >= f_min[j], <= f_max[j];   # Decision variable: amount of each food to buy within bounds

minimize TotalCost:                     # Objective: minimize total food cost
    sum {j in FOOD} cost[j] * Buy[j];

subject to DietMin {i in MINREQ}:       # Minimum nutrient constraints (must satisfy lower bounds)
    sum {j in FOOD} amt[i,j] * Buy[j] >= n_min[i];

subject to DietMax {i in MAXREQ}:       # Maximum nutrient constraints (must satisfy upper bounds)
    sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

In [ ]:
%%ampl_eval
data;

# Nutrient requirement sets
set MINREQ := protein fat calcium;
set MAXREQ := vitamin iron calcium;

# Foods
set FOOD := bread milk apple meat;

# Nutrient content per food unit
param amt:        bread  milk  apple  meat :=
    protein        10     8      2     25
    fat             5     3      1     15
    calcium        20    15      5     10
    vitamin         2     6     10      1
    iron            1     2      3      8 ;

# Minimum nutrient requirements
param n_min := protein 50, fat 20, calcium 30;

# Maximum nutrient limits
param n_max := calcium 60, vitamin 50, iron 40;

# Food costs
param cost := bread 2, milk 3, apple 1.5, meat 5;

option solver highs;                # choose HiGHS linear solver
solve;                              # solve the optimization model
display Buy, TotalCost;             # show optimal food quantities and total cost
display DietMax.dual, DietMin.dual; # display shadow prices (dual values) of constraints

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 10
2 simplex iterations
0 barrier iterations
Buy [*] :=
apple  0
bread  0.625
 meat  1.75
 milk  0
;

TotalCost = 10

:       DietMax.dual DietMin.dual    :=
calcium       0           0
fat            .          0
iron          0             .
protein        .          0.2
vitamin       0             .
;



#### amplpy Formulation

In [7]:
import pandas as pd  # use pandas for structured tabular data input

# Define nutrient sets (lower and upper bound groups)
ampl.set['MINREQ'] = ['protein', 'fat', 'calcium']
ampl.set['MAXREQ'] = ['vitamin', 'iron', 'calcium']
ampl.set['FOOD'] = ['bread', 'milk', 'apple', 'meat']

amt = pd.DataFrame({                # Nutrient-content matrix (rows = nutrients, columns = foods)
    'bread': [10, 5, 20, 2, 1],
    'milk':  [8, 3, 15, 6, 2],
    'apple': [2, 1, 5, 10, 3],
    'meat':  [25, 15, 10, 1, 8]
}, index=['protein', 'fat', 'calcium', 'vitamin', 'iron'])

ampl.param['amt'].set_values(amt)   # Load nutrient matrix into AMPL

ampl.param['n_min'].set_values({    # Minimum nutrient requirements
    'protein': 50,
    'fat': 20,
    'calcium': 30})

ampl.param['n_max'].set_values({    # Maximum nutrient limit
    'calcium': 60,
    'vitamin': 50,
    'iron': 40})

ampl.param['cost'].set_values({     # Food cost
    'bread': 2,
    'milk': 3,
    'apple': 1.5,
    'meat': 5})

# Solve optimization model
ampl.set_option('solver', 'highs')
ampl.solve()

buy = ampl.var['Buy'].get_values().to_dict()    # Extract optimal solution

# Print decision variables and objective value
print("Buy =", {k: round(v, 3) for k, v in buy.items()})
print("TotalCost =", ampl.obj['TotalCost'].value())

# Access constraint dual values (shadow prices)
DietMax = ampl.get_constraint('DietMax')
DietMin = ampl.get_constraint('DietMin')

# Print dual values for interpretation of constraints
print("DietMax duals =", {i: DietMax[i].dual() for i in ampl.set['MAXREQ']})
print("DietMin duals =", {i: DietMin[i].dual() for i in ampl.set['MINREQ']})

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 10
0 simplex iterations
0 barrier iterations
Buy = {'apple': 0, 'bread': 0.625, 'meat': 1.75, 'milk': 0}
TotalCost = 10.0
DietMax duals = {'vitamin': 0.0, 'iron': 0.0, 'calcium': 0.0}
DietMin duals = {'protein': 0.2, 'fat': 0.0, 'calcium': 0.0}
